# Analyse du discours du corpus TASS : environnements lexicaux et structures syntaxiques

Ce notebook prolonge la constitution du corpus TASS par une exploration du discours produit autour des déplacements forcés d’Ukraine vers la Russie. 

L’objectif est d’observer comment certains termes-cibles — notamment « беженец » / réfugié, « волонтер » / bénévole, « украина » / Ukraine et « россия » / Russie — sont insérés dans le corpus : avec quels mots ils coapparaissent, de quels champs lexicaux ils se rapprochent, et dans quelles relations syntaxiques ils apparaissent.

Le notebook combine quatre opérations principales : lemmatisation des textes, analyse des cooccurrences et des proximités lexicales avec Word2Vec, puis extraction de patrons syntaxiques avec spaCy.


In [1]:
import json
from pathlib import Path
import pandas as pd

## 1. Chargement du corpus nettoyé

Cette première étape charge le corpus TASS nettoyé, contenant les métadonnées principales des articles ainsi que leur texte intégral. Le fichier JSON constitue le point de départ de l’analyse lexicale et syntaxique menée dans ce notebook.


In [6]:
# Chemin vers le fichier JSON
json_path = Path("tass_clean_with_text.json")

# Chargement du JSON
with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Nombre d'éléments : {len(data) if hasattr(data, '__len__') else 'N/A'}")


Nombre d'éléments : 1870


## 2. Lemmatisation des textes

Avant d’analyser les cooccurrences et les proximités lexicales, les textes sont normalisés par lemmatisation. Cette opération réduit les formes fléchies à leur lemme, ce qui permet de regrouper des variantes grammaticales d’un même mot.

La lemmatisation est effectuée avec le modèle russe de spaCy. Le résultat est enregistré dans un fichier JSONL afin de pouvoir reprendre l’analyse sans relancer toute l’étape linguistique.


In [7]:
import re
from collections import Counter, defaultdict
import spacy 
from pathlib import Path
from tqdm.auto import tqdm

In [8]:
nlp = spacy.load("ru_core_news_sm")

In [9]:
OUTPUT_LEMMA_FILE = Path("tass_text_lemma.jsonl")
SAVE_EVERY = 500

In [10]:
def lemmatize_texte_spacy(texte: str) -> str:
    doc = nlp(str(texte))
    lemmas = [t.lemma_.lower() for t in doc if not t.is_space]
    return " ".join(lemmas)

n_total = len(data) if hasattr(data, "__len__") else None
n_done = 0
n_missing = 0

with OUTPUT_LEMMA_FILE.open("w", encoding="utf-8") as f_out:
    for i, item in enumerate(tqdm(data, total=n_total, desc="Lemmatisation"), start=1):
        out_item = dict(item)
        text_value = str(item.get("text", "")).strip()

        if text_value:
            out_item["texte_lemmatise"] = lemmatize_texte_spacy(text_value)
            n_done += 1
        else:
            n_missing += 1

        f_out.write(json.dumps(out_item, ensure_ascii=False) + "\n")

        # autosauvegarde périodique pour limiter les pertes en cas d'arrêt
        if i % SAVE_EVERY == 0:
            f_out.flush()

print(f"Terminé. Fichier créé : {OUTPUT_LEMMA_FILE}")
print(f"Éléments traités : {n_done}/{n_total if n_total is not None else 'N/A'}")
print(f"Éléments sans clé 'text' (ou vide) : {n_missing}")

Lemmatisation:   0%|          | 0/1870 [00:00<?, ?it/s]

Terminé. Fichier créé : tass_text_lemma.jsonl
Éléments traités : 1870/1870
Éléments sans clé 'text' (ou vide) : 0


## 3. Analyse des cooccurrences autour de termes-cibles

Cette section recharge le corpus lemmatisé et mesure les mots qui apparaissent dans le voisinage immédiat de plusieurs termes-cibles. L’objectif est d’identifier les environnements lexicaux associés à chaque notion dans le discours de TASS.

L’analyse porte ici sur quatre cibles : réfugiés, bénévoles, Ukraine et Russie.


In [12]:
import json

data = []
with open("tass_text_lemma.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

In [13]:
import re
from collections import Counter
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm

### 3.1. Définition des paramètres de cooccurrence

La fenêtre de cooccurrence est fixée à cinq mots à gauche et cinq mots à droite du terme-cible. Les mots-outils et certains termes trop génériques sont exclus afin de faire ressortir les associations lexicales les plus interprétables.


In [14]:
WINDOW = 5
TOP_K = 30

TARGETS = {
    "Refugies": "беженец",
    "Benevoles": "волонтер",
    "Ukraine": "украина",
    "Russie": "россия",
}

stop_words = set(w.replace("ё", "е") for w in nlp.Defaults.stop_words)
stop_words.update({"это", "который", "также", "весь", "тысяча"})

### 3.2. Calcul et visualisation des cooccurrences

Pour chaque occurrence d’un terme-cible, le code collecte les mots présents dans la fenêtre définie, compte leur fréquence, puis affiche les trente cooccurrents les plus fréquents.

Les graphiques permettent de comparer les environnements lexicaux associés aux différentes cibles. Ils servent de support à l’interprétation qualitative plutôt que de preuve statistique autonome.


In [15]:
def norm_token(t: str) -> str:
    return str(t).lower().replace("ё", "е").strip()

def get_tokens_from_item(item: dict) -> list[str]:
    if item.get("texte_lemmatise"):
        raw = str(item.get("texte_lemmatise"))
    else:
        raw = lemmatize_texte_spacy(str(item.get("text", "")))

    toks = [norm_token(t) for t in raw.split()]
    toks = [t for t in toks if re.fullmatch(r"[a-zа-я]+(?:-[a-zа-я]+)?", t)]
    return toks

context_counters = {name: Counter() for name in TARGETS}
target_occurrences = {name: 0 for name in TARGETS}

for item in tqdm(data, desc="Collecte des cooccurrences"):
    tokens = get_tokens_from_item(item)
    n = len(tokens)
    if n == 0:
        continue

    for target_name, target_lemma in TARGETS.items():
        target_lemma = norm_token(target_lemma)
        for i, tok in enumerate(tokens):
            if tok != target_lemma:
                continue

            target_occurrences[target_name] += 1
            left = max(0, i - WINDOW)
            right = min(n, i + WINDOW + 1)

            for j in range(left, right):
                if j == i:
                    continue
                cand = tokens[j]
                if cand in stop_words or cand == target_lemma or len(cand) < 2:
                    continue
                context_counters[target_name][cand] += 1

rows = []
for target_name, cnt in context_counters.items():
    for rank, (word, freq) in enumerate(cnt.most_common(TOP_K), start=1):
        rows.append({
            "target": target_name,
            "rank": rank,
            "word": word,
            "freq": freq,
            "target_occurrences": target_occurrences[target_name],
        })

top_df = pd.DataFrame(rows)
target_order = list(TARGETS.keys())

if top_df.empty:
    print("Aucune donnée à visualiser.")
else:
    max_freq_global = float(top_df["freq"].max())
    x_max = max_freq_global * 1.1 if max_freq_global > 0 else 1.0

    for target_name in target_order:
        sub = top_df[top_df["target"] == target_name].copy()
        print("\n" + "=" * 70)
        print(f"Cible : {target_name} | Occurrences trouvées : {target_occurrences[target_name]}")

        if sub.empty:
            print("Aucune cooccurrence trouvée.")
            continue

        display(sub[["rank", "word", "freq"]])

        sub = sub.sort_values("freq", ascending=True)
        fig = px.bar(
            sub,
            x="freq",
            y="word",
            orientation="h",
            color="freq",
            color_continuous_scale="Sunset",
            range_x=[0, x_max],
            range_color=[0, max_freq_global] if max_freq_global > 0 else None,
            hover_data={
                "target": True,
                "rank": True,
                "freq": True,
            },
            title=f"Cible : {target_name}",
            height=650,
            width=1200,
        )

        fig.update_yaxes(automargin=True, tickfont=dict(size=11), title="Cooccurrences")
        fig.update_xaxes(title="Fréquence (échelle unique)")
        fig.update_layout(
            template="plotly_white",
            margin=dict(l=180, r=100, t=70, b=30),
            coloraxis_colorbar=dict(title="Fréquence", x=1.02),
            showlegend=False,
        )
        fig.show()

    print("Couleur: plus la barre est foncée, plus la cooccurrence est fréquente.")

Collecte des cooccurrences:   0%|          | 0/1870 [00:00<?, ?it/s]


Cible : Refugies | Occurrences trouvées : 1544


,rank,word,freq
0,1,украина,222
1,2,регион,208
2,3,прибыть,183
3,4,помощь,182
4,5,прибывать,168
5,6,донбасс,151
6,7,рф,138
7,8,российский,129
8,9,размещение,127
9,10,поручение,126



Cible : Benevoles | Occurrences trouvées : 327


,rank,word,freq
30,1,помощь,55
31,2,житель,35
32,3,помогать,34
33,4,область,32
34,5,человек,30
35,6,гуманитарный,29
36,7,работа,26
37,8,центр,26
38,9,работать,24
39,10,медик,23



Cible : Ukraine | Occurrences trouvées : 2152


,rank,word,freq
60,1,военный,370
61,2,операция,358
62,3,специальный,345
63,4,территория,328
64,5,проведение,309
65,6,донбасс,298
66,7,россия,248
67,8,республика,230
68,9,беженец,222
69,10,человек,216



Cible : Russie | Occurrences trouvées : 2841


,rank,word,freq
90,1,февраль,582
91,2,область,550
92,3,ростовский,494
93,4,президент,456
94,5,владимир,433
95,6,территория,430
96,7,путин,428
97,8,республика,386
98,9,житель,383
99,10,сутки,365


Couleur: plus la barre est foncée, plus la cooccurrence est fréquente.


### 3.3. Sauvegarde des résultats de cooccurrence

Les tableaux de cooccurrences sont sauvegardés dans un fichier CSV. Cette sauvegarde permet de réutiliser les résultats sans recalculer l’ensemble du corpus et de les intégrer ensuite dans une analyse manuelle.


In [16]:
top_df.to_csv("discours_analysis_support/cooccurrences.csv", index=False)

### 3.4. Extraction d’exemples contextualisés

Les cooccurrences seules ne suffisent pas à interpréter le sens des associations lexicales. Cette étape extrait donc plusieurs passages courts autour des couples `terme-cible / cooccurrent` afin de revenir aux formulations concrètes des articles.


In [15]:
#Exemples
N_EXAMPLES = 5
WINDOW_EX = 8  # taille du contexte affiché autour de la cible

In [18]:
if "top_df" not in globals() or top_df.empty:
    raise ValueError("top_df est vide ou absent. Exécute d'abord la cellule de cooccurrence.")

# mots à documenter par cible (ceux du top cooccurrence)
words_by_target = {
    target: set(top_df.loc[top_df["target"] == target, "word"].tolist())
    for target in TARGETS.keys()
}

# stockage des exemples
examples = defaultdict(list)   # clé: (target, word) -> liste d'extraits
seen = defaultdict(set)        # pour éviter les doublons exacts

def norm_lemma(x: str) -> str:
    return str(x).lower().replace("ё", "е").strip()

for item in tqdm(data, desc="Extraction des exemples"):
    text = str(item.get("text", "")).strip()
    if not text:
        continue

    doc = nlp(text)
    lemmas = [norm_lemma(t.lemma_) for t in doc]
    surfaces = [t.text for t in doc]
    n = len(lemmas)

    if n == 0:
        continue

    for target_name, target_lemma in TARGETS.items():
        tgt = norm_lemma(target_lemma)
        candidate_words = words_by_target.get(target_name, set())
        if not candidate_words:
            continue

        for i, lem in enumerate(lemmas):
            if lem != tgt:
                continue

            left = max(0, i - WINDOW)
            right = min(n, i + WINDOW + 1)

            for j in range(left, right):
                if j == i:
                    continue

                w = lemmas[j]
                if w not in candidate_words:
                    continue
                if len(examples[(target_name, w)]) >= N_EXAMPLES:
                    continue

                # extrait autour de la cible (surface texte, plus lisible)
                ex_l = max(0, i - WINDOW_EX)
                ex_r = min(n, i + WINDOW_EX + 1)
                excerpt = " ".join(surfaces[ex_l:ex_r]).strip()

                # dédoublonnage
                key = excerpt.lower()
                if key in seen[(target_name, w)]:
                    continue

                seen[(target_name, w)].add(key)
                examples[(target_name, w)].append({
                    "cible": target_name,
                    "co_word": w,
                    "excerpt": excerpt,
                    "title": str(item.get("title", "")),
                    "url": str(item.get("url", "")),
                })

# assembler en table
rows = []
for (target_name, w), ex_list in examples.items():
    for k, ex in enumerate(ex_list, start=1):
        rows.append({
            "cible": target_name,
            "co_word": w,
            "example_id": k,
            "excerpt": ex["excerpt"],
            "title": ex["title"],
            "url": ex["url"],
        })

examples_df = pd.DataFrame(rows)


Extraction des exemples:   0%|          | 0/1870 [00:00<?, ?it/s]

In [21]:
examples_df.to_csv("discours_analysis_support/exemples_cooccurrence.csv", index=False)

### 3.5. Rechargement des cooccurrences et des exemples

Cette cellule recharge les fichiers CSV produits précédemment. Elle permet de reprendre l’analyse directement à partir des résultats sauvegardés, sans relancer la lemmatisation ni l’extraction des exemples.


In [17]:
examples_df = pd.read_csv("discours_analysis_support/exemples_cooccurrence.csv")
top_df = pd.read_csv("discours_analysis_support/cooccurrences.csv")

In [18]:

examples_df = examples_df.sort_values(["cible", "co_word", "example_id"]).reset_index(drop=True)
print(f"Total exemples extraits : {len(examples_df)}")

    

Total exemples extraits : 595


In [19]:
import math
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

### 3.6. Exploration interactive des exemples

L’interface suivante permet de filtrer les extraits par terme-cible et par cooccurrent. Elle sert à passer de la vue quantitative — les fréquences — à une lecture qualitative des passages où les associations lexicales apparaissent.


In [28]:
import math
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

df_view = examples_df.copy()
df_view["url"] = df_view["url"].fillna("").astype(str)
df_view["url_clickable"] = df_view["url"].apply(
    lambda u: f'<a href="{u}" target="_blank">ouvrir</a>' if u.strip() else ""
)

if "top_df" not in globals() or top_df.empty:
    raise ValueError("top_df est vide ou absent. Exécute d'abord la cellule de cooccurrence.")

ref_df = top_df[["target", "word", "freq"]].copy()
ref_map = {(r["target"], r["word"]): int(r["freq"]) for _, r in ref_df.iterrows()}

df_view["_ref_freq"] = df_view.apply(
    lambda r: ref_map.get((r["cible"], r["co_word"]), 0), axis=1
)

df_view = df_view[["cible", "co_word", "example_id", "excerpt", "title", "url_clickable", "_ref_freq"]]

cible_dd = widgets.Dropdown(
    options=["Toutes"] + sorted(df_view["cible"].dropna().unique().tolist()),
    value="Toutes",
    description="Cible:",
    layout=widgets.Layout(width="260px"),
)

word_dd = widgets.Dropdown(
    options=["Tous"],
    value="Tous",
    description="Mot:",
    layout=widgets.Layout(width="320px"),
)

page_size_dd = widgets.Dropdown(
    options=[10, 20, 30, 50, 100],
    value=20,
    description="Lignes:",
    layout=widgets.Layout(width="170px"),
)

page_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="Page:",
    continuous_update=False,
    layout=widgets.Layout(width="350px"),
)

table_html = widgets.HTML()
info_html = widgets.HTML()

table_box = widgets.Box(
    [table_html],
    layout=widgets.Layout(
        width="100%",
        overflow="auto",
        border="1px solid #ddd",
        padding="6px"
    )
)

updating_ui = False

def get_filtered_df():
    sub = df_view.copy()

    if cible_dd.value != "Toutes":
        sub = sub[sub["cible"] == cible_dd.value]

    if word_dd.value != "Tous":
        sub = sub[sub["co_word"] == word_dd.value]

    sub = sub.sort_values(
        ["_ref_freq", "co_word", "example_id"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    return sub

def refresh_word_options():
    global updating_ui
    updating_ui = True
    try:
        if cible_dd.value == "Toutes":
            counts = (
                ref_df.groupby("word", as_index=False)["freq"]
                .sum()
                .sort_values("freq", ascending=False)
            )
            available = set(df_view["co_word"].dropna().unique().tolist())
            words = [w for w in counts["word"].tolist() if w in available]
        else:
            counts = (
                ref_df[ref_df["target"] == cible_dd.value]
                .sort_values("freq", ascending=False)
            )
            available = set(
                df_view.loc[df_view["cible"] == cible_dd.value, "co_word"].dropna().unique().tolist()
            )
            words = [w for w in counts["word"].tolist() if w in available]

        prev = word_dd.value
        new_options = ["Tous"] + words
        word_dd.options = new_options
        word_dd.value = prev if prev in new_options else "Tous"
        page_slider.value = 1
    finally:
        updating_ui = False

def render():
    global updating_ui
    if updating_ui:
        return

    sub = get_filtered_df()

    page_size = int(page_size_dd.value)
    total = len(sub)
    total_pages = max(1, math.ceil(total / page_size))

    updating_ui = True
    try:
        page_slider.max = total_pages
        if page_slider.value > total_pages:
            page_slider.value = total_pages
    finally:
        updating_ui = False

    p = page_slider.value
    start = (p - 1) * page_size
    end = start + page_size
    page_df = sub.iloc[start:end].copy()

    if "_ref_freq" in page_df.columns:
        page_df = page_df.drop(columns=["_ref_freq"])

    if page_df.empty:
        table_html.value = "<i>Aucune ligne pour ce filtre.</i>"
        info_html.value = ""
        return

    html_table = page_df.to_html(index=False, escape=False)
    table_html.value = html_table
    info_html.value = f"Lignes affichées : {start + 1}-{min(end, total)} / {total} | Page {p}/{total_pages}"

def on_cible_change(change):
    if updating_ui:
        return
    refresh_word_options()
    render()

def on_filter_change(change):
    if updating_ui:
        return
    render()

cible_dd.observe(on_cible_change, names="value")
word_dd.observe(on_filter_change, names="value")
page_size_dd.observe(on_filter_change, names="value")
page_slider.observe(on_filter_change, names="value")

controls_row1 = widgets.HBox([cible_dd, word_dd, page_size_dd])
controls_row2 = widgets.HBox([page_slider])

ui = widgets.VBox([controls_row1, controls_row2, table_box, info_html])

refresh_word_options()
render()
display(ui)

## 4. Analyse des proximités lexicales avec Word2Vec

La section suivante entraîne un modèle Word2Vec sur le corpus lemmatisé. Contrairement aux cooccurrences simples, Word2Vec permet d’identifier des mots qui apparaissent dans des contextes similaires, même lorsqu’ils ne sont pas immédiatement voisins dans les textes.

Cette approche sert à repérer des champs sémantiques et des proximités discursives autour des mêmes termes-cibles.


In [35]:
from gensim.models import Word2Vec
import pandas as pd
import random

### 4.1. Paramètres du modèle Word2Vec

Les paramètres définissent la taille des vecteurs, la fenêtre contextuelle, le seuil minimal de fréquence des mots et le nombre d’époques d’entraînement. Le modèle utilise ici l’architecture skip-gram, adaptée à l’identification de relations lexicales dans des corpus spécialisés.


In [36]:
W2V_VECTOR_SIZE = 100
W2V_WINDOW = 5
W2V_MIN_COUNT = 3
W2V_SG = 1
W2V_EPOCHS = 15
W2V_TOPN = 20
RANDOM_SEED = 42

### 4.2. Définition des termes-cibles pour Word2Vec

Les mêmes termes-cibles sont repris afin de comparer les résultats obtenus par cooccurrences directes et par proximités vectorielles.


In [37]:
TARGETS_W2V = {
    "Refugies": "беженец",
    "Benevoles": "волонтер",
    "Ukraine": "украина",
    "Russie": "россия",
}

### 4.3. Préparation des tokens pour l’entraînement

Cette cellule normalise les tokens utilisés par Word2Vec. Elle privilégie les textes déjà lemmatisés et applique un filtrage simple afin de conserver principalement des formes lexicales exploitables.


In [38]:
def normalize_token(x: str) -> str:
    return str(x).lower().replace("ё", "е").strip()

def get_lemmas_for_w2v(item: dict) -> list[str]:
    # utilise d'abord la version déjà lemmatisée, sinon fallback spaCy
    if item.get("texte_lemmatise"):
        raw = str(item.get("texte_lemmatise"))
        toks = [normalize_token(t) for t in raw.split()]
    else:
        txt = str(item.get("text", "")).strip()
        if not txt:
            return []
        doc = nlp(txt)
        toks = [normalize_token(t.lemma_) for t in doc if not t.is_space]

    # garde les tokens alphabétiques russes/latins (optionnellement avec tiret)
    toks = [t for t in toks if re.fullmatch(r"[a-zа-я]+(?:-[a-zа-я]+)?", t)]
    return toks

### 4.4. Construction du corpus d’entraînement

Le corpus est transformé en une liste de listes de tokens, format attendu par Word2Vec. Chaque article devient une séquence de lemmes utilisée pour apprendre les proximités lexicales.


In [39]:
# 1) Préparation du corpus
sentences = []
for item in tqdm(data, desc="Préparation corpus Word2Vec"):
    toks = get_lemmas_for_w2v(item)
    if toks:
        sentences.append(toks)

print(f"Phrases utilisées: {len(sentences)}")
if not sentences:
    raise ValueError("Corpus vide pour Word2Vec.")

#преобразует данные в формат для Word2Vec: список предложений list[list[str]] (sentences);
#чистит/нормализует токены (lower, ё→е, regex-фильтр);
#пропускает пустые записи;
#только если texte_lemmatise отсутствует, делает fallback через spaCy.

Préparation corpus Word2Vec:   0%|          | 0/1870 [00:00<?, ?it/s]

Phrases utilisées: 1870


### 4.5. Entraînement du modèle Word2Vec

Le modèle est entraîné directement sur le corpus TASS. Le vocabulaire final correspond aux mots suffisamment fréquents pour être conservés selon le seuil défini plus haut.


In [40]:
# 2) Entraînement Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=W2V_VECTOR_SIZE,
    window=W2V_WINDOW,
    min_count=W2V_MIN_COUNT,
    workers=4,
    sg=W2V_SG,
    seed=RANDOM_SEED,
    epochs=W2V_EPOCHS,
)

print(f"Vocabulaire Word2Vec: {len(w2v_model.wv)}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulaire Word2Vec: 5367


### 4.6. Extraction des mots les plus proches des termes-cibles

Pour chaque terme-cible, le modèle retourne les voisins vectoriels les plus proches. Ces listes permettent d’identifier les termes qui partagent des contextes d’apparition similaires dans le corpus.


In [41]:
# 3) Most similar — 1ère table Word2Vec (similar_df)
if "w2v_model" not in globals():
    raise ValueError("w2v_model absent. Exécute d'abord l'entraînement Word2Vec.")
if "TARGETS_W2V" not in globals():
    raise ValueError("TARGETS_W2V absent.")
if "W2V_TOPN" not in globals():
    W2V_TOPN = 20
if "normalize_token" not in globals():
    def normalize_token(x: str) -> str:
        return str(x).lower().replace("ё", "е").strip()

sim_rows = []
for label, term in TARGETS_W2V.items():
    t = normalize_token(term)
    if t not in w2v_model.wv:
        print(f"Terme absent du vocabulaire: {label} -> {t}")
        continue

    for rank, (word, score) in enumerate(w2v_model.wv.most_similar(t, topn=W2V_TOPN), start=1):
        sim_rows.append({
            "target": label,
            "target_word": t,
            "rank": rank,
            "similar_word": word,
            "similarity": float(score),
        })

similar_df = pd.DataFrame(sim_rows).sort_values(["target", "rank"]).reset_index(drop=True)

if similar_df.empty:
    print("Aucun voisin trouvé pour les cibles.")
else:
    display(similar_df)

,target,target_word,rank,similar_word,similarity
0,Benevoles,волонтер,1,доброволец,0.644791
1,Benevoles,волонтер,2,петербуржец,0.636546
2,Benevoles,волонтер,3,волонтерский,0.620946
3,Benevoles,волонтер,4,вскс,0.620145
4,Benevoles,волонтер,5,рота,0.608519
...,...,...,...,...,...
75,Ukraine,украина,16,выплачиваться,0.496406
76,Ukraine,украина,17,украинскую,0.496240
77,Ukraine,украина,18,вдвое,0.494725
78,Ukraine,украина,19,профинансировать,0.493567


### 4.7. Sauvegarde des voisins Word2Vec

Les résultats Word2Vec sont sauvegardés sous forme de tableau CSV afin de pouvoir les consulter, les visualiser ou les intégrer dans d’autres étapes de l’analyse.


In [42]:
similar_df.to_csv("discours_analysis_support/word2vec.csv", index=False)

### 4.8. Rechargement des résultats Word2Vec

Cette cellule recharge les voisins vectoriels déjà calculés. Elle permet de poursuivre l’analyse à partir des résultats enregistrés, sans réentraîner le modèle.


In [43]:
similar_df = pd.read_csv("discours_analysis_support/word2vec.csv")

### 4.9. Table interactive des proximités lexicales

L’interface permet de parcourir les voisins Word2Vec par terme-cible, de filtrer les résultats et d’examiner les scores de similarité. Elle facilite la sélection des associations lexicales les plus pertinentes pour l’interprétation qualitative.


In [44]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# копия таблицы
df = similar_df.copy()

# приведение типов
if "rank" in df.columns:
    df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
if "similarity" in df.columns:
    df["similarity"] = pd.to_numeric(df["similarity"], errors="coerce")

# widgets
target_options = ["Tous"] + sorted(df["target"].dropna().astype(str).unique().tolist())

target_dd = widgets.Dropdown(
    options=target_options,
    value="Tous",
    description="Target:",
    layout=widgets.Layout(width="260px")
)

rank_min = int(df["rank"].min()) if "rank" in df.columns else 1
rank_max = int(df["rank"].max()) if "rank" in df.columns else 20

rank_slider = widgets.IntRangeSlider(
    value=[rank_min, rank_max],
    min=rank_min,
    max=rank_max,
    step=1,
    description="Rank:",
    continuous_update=False,
    layout=widgets.Layout(width="420px")
)

sim_min = float(df["similarity"].min()) if "similarity" in df.columns else 0.0
sim_max = float(df["similarity"].max()) if "similarity" in df.columns else 1.0

sim_slider = widgets.FloatSlider(
    value=sim_min,
    min=sim_min,
    max=sim_max,
    step=0.01,
    description="Sim ≥",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

word_text = widgets.Text(
    value="",
    placeholder="ex. réfugié",
    description="Mot:",
    layout=widgets.Layout(width="320px")
)

n_rows = widgets.IntSlider(
    value=30,
    min=5,
    max=200,
    step=5,
    description="Lignes:",
    continuous_update=False,
    layout=widgets.Layout(width="260px")
)

sort_by = widgets.Dropdown(
    options=df.columns.tolist(),
    value="rank" if "rank" in df.columns else df.columns[0],
    description="Trier:",
    layout=widgets.Layout(width="260px")
)

ascending_cb = widgets.Checkbox(
    value=True,
    description="Croissant"
)

table_html = widgets.HTML()
info_html = widgets.HTML()

table_box = widgets.Box(
    [table_html],
    layout=widgets.Layout(
        width="100%",
        overflow="auto",
        border="1px solid #ddd",
        padding="6px"
    )
)

updating_ui = False

def get_filtered_df():
    filtered = df.copy()

    if target_dd.value != "Tous":
        filtered = filtered[filtered["target"].astype(str) == target_dd.value]

    if "rank" in filtered.columns:
        filtered = filtered[
            filtered["rank"].between(rank_slider.value[0], rank_slider.value[1])
        ]

    if "similarity" in filtered.columns:
        filtered = filtered[filtered["similarity"] >= sim_slider.value]

    if word_text.value.strip() and "similar_word" in filtered.columns:
        q = word_text.value.strip().lower()
        filtered = filtered[
            filtered["similar_word"].astype(str).str.lower().str.contains(q, na=False)
        ]

    filtered = filtered.sort_values(by=sort_by.value, ascending=ascending_cb.value)
    filtered = filtered.head(n_rows.value).reset_index(drop=True)

    return filtered

def render(change=None):
    global updating_ui
    if updating_ui:
        return

    filtered = get_filtered_df()

    if filtered.empty:
        table_html.value = "<i>Aucune ligne pour ce filtre.</i>"
        info_html.value = ""
        return

    table_html.value = filtered.to_html(index=False)
    info_html.value = f"Lignes affichées : {len(filtered)}"

for w in [target_dd, rank_slider, sim_slider, word_text, n_rows, sort_by, ascending_cb]:
    w.observe(render, names="value")

controls = widgets.VBox([
    widgets.HBox([target_dd, word_text]),
    widgets.HBox([rank_slider, sim_slider]),
    widgets.HBox([sort_by, ascending_cb, n_rows]),
])

ui = widgets.VBox([controls, table_box, info_html])

render()
display(ui)

### 4.10. Visualisation en réseau des voisins Word2Vec

Cette cellule représente chaque terme-cible et ses voisins sous forme de graphe. L’épaisseur des liens traduit le score de similarité. La visualisation aide à repérer les champs lexicaux dominants autour de chaque cible.


In [51]:
# 5) Petit network graph par target (mot cible + voisins Word2Vec)
import networkx as nx
import plotly.graph_objects as go

if "similar_df" not in globals() or similar_df.empty:
    raise ValueError("similar_df est vide. Exécute d'abord la cellule 3) Most similar.")

NET_TOPN = 10  # nombre de voisins par cible

for label, term in TARGETS_W2V.items():
    t = normalize_token(term)
    if t not in w2v_model.wv:
        print(f"Cible ignorée (absente du vocabulaire): {label} -> {t}")
        continue

    sub = (
        similar_df[similar_df["target"] == label]
        .sort_values("rank")
        .head(NET_TOPN)
    )

    if sub.empty:
        print(f"Aucun voisin à tracer pour {label}.")
        continue

    # Graphe étoile: cible au centre + voisins
    G = nx.Graph()
    G.add_node(t, node_type="cible")

    for _, r in sub.iterrows():
        w = str(r["similar_word"])
        sim = float(r["similarity"])
        G.add_node(w, node_type="voisin")
        G.add_edge(t, w, weight=sim)

    # Layout
    pos = nx.spring_layout(G, seed=RANDOM_SEED, k=0.9)

    # Arêtes avec épaisseur proportionnelle à la similarité
    sim_values = [G.edges[u, v]["weight"] for u, v in G.edges()]
    sim_min, sim_max = min(sim_values), max(sim_values)

    def edge_width(sim: float, w_min: float = 1.0, w_max: float = 8.0) -> float:
        if sim_max == sim_min:
            return (w_min + w_max) / 2
        return w_min + (sim - sim_min) * (w_max - w_min) / (sim_max - sim_min)

    edge_traces = []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        sim = float(G.edges[u, v]["weight"])

        edge_traces.append(
            go.Scatter(
                x=[x0, x1],
                y=[y0, y1],
                mode="lines",
                line=dict(width=edge_width(sim), color="#9aa0a6"),
                hovertemplate=f"similarity={sim:.3f}<extra>lien</extra>",
                showlegend=False,
            )
        )

    # Nœuds (cible)
    target_trace = go.Scatter(
        x=[pos[n][0] for n in G.nodes() if G.nodes[n]["node_type"] == "cible"],
        y=[pos[n][1] for n in G.nodes() if G.nodes[n]["node_type"] == "cible"],
        mode="markers+text",
        text=[n for n in G.nodes() if G.nodes[n]["node_type"] == "cible"],
        textposition="top center",
        marker=dict(size=24, color="#d62728", line=dict(width=1, color="white")),
        name="cible",
        hovertemplate="mot=%{text}<extra>cible</extra>",
    )

    # Nœuds (voisins)
    voisin_nodes = [n for n in G.nodes() if G.nodes[n]["node_type"] == "voisin"]
    voisin_sizes = []
    for n in voisin_nodes:
        sim = G.edges[t, n]["weight"] if G.has_edge(t, n) else 0.0
        voisin_sizes.append(10 + sim * 18)

    voisin_trace = go.Scatter(
        x=[pos[n][0] for n in voisin_nodes],
        y=[pos[n][1] for n in voisin_nodes],
        mode="markers+text",
        text=voisin_nodes,
        textposition="top center",
        marker=dict(size=voisin_sizes, color="#1f77b4", line=dict(width=0.8, color="white")),
        name="voisin",
        hovertemplate="mot=%{text}<extra>voisin</extra>",
    )

    fig = go.Figure(data=edge_traces + [voisin_trace, target_trace])
    fig.update_layout(
        title=f"Network Word2Vec — {label} ({t})",
        template="plotly_white",
        width=980,
        height=680,
        showlegend=True,
        margin=dict(l=20, r=20, t=60, b=20),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()

print("Terminé: network graph affiché pour chaque target (épaisseur des liens = similarity).")

Terminé: network graph affiché pour chaque target (épaisseur des liens = similarity).


### 4.11. Validation contextuelle des voisins Word2Vec

Les proximités vectorielles doivent être vérifiées dans les textes. Cette section prépare donc une extraction d’exemples pour les termes-cibles et leurs principaux voisins, afin d’observer les usages concrets dans les articles.


In [52]:
# 6) Validation contextuelle des voisins Word2Vec (visualisation simple)
from collections import defaultdict
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd
from tqdm.auto import tqdm

TOP_NEIGHBORS_FOR_CONTEXT = 10
EXAMPLES_PER_WORD = 5

def _norm(x: str) -> str:
    if "normalize_token" in globals():
        return normalize_token(x)
    return str(x).lower().replace("ё", "е").strip()

### 4.12. Constitution des exemples pour les voisins Word2Vec

Le code parcourt les phrases du corpus et extrait plusieurs exemples contenant soit le terme-cible, soit un de ses voisins Word2Vec. Ces extraits permettent de vérifier si la proximité calculée correspond à une association interprétable dans le discours.


In [53]:
if "similar_df" not in globals() or similar_df.empty:
    raise ValueError("similar_df est vide. Exécute d'abord la cellule 3) Most similar.")
if "w2v_model" not in globals():
    raise ValueError("w2v_model est absent. Exécute d'abord l'entraînement Word2Vec.")

# 1) top-10 voisins par target
neighbors_by_target = {}
target_words = {}
for label, term in TARGETS_W2V.items():
    tgt = _norm(term)
    target_words[label] = tgt
    if tgt not in w2v_model.wv:
        neighbors_by_target[label] = []
        continue
    sub = (
        similar_df[similar_df["target"] == label]
        .sort_values("rank")
        .head(TOP_NEIGHBORS_FOR_CONTEXT)
    )
    neighbors_by_target[label] = [str(w) for w in sub["similar_word"].tolist()]

required_terms = set()
for label, tgt in target_words.items():
    required_terms.add(tgt)
    required_terms.update(neighbors_by_target.get(label, []))

# 2) extraction des phrases utiles
sentence_records = []
for item in tqdm(data, desc="Scan phrases (target + voisins)"):
    txt = str(item.get("text", "")).strip()
    if not txt:
        continue

    doc = nlp(txt)
    for sent in doc.sents:
        tokens = [t for t in sent if not t.is_space and not t.is_punct]
        if not tokens:
            continue

        lemmas = {_norm(t.lemma_) for t in tokens}
        if not (lemmas & required_terms):
            continue

        sentence_records.append({
            "title": str(item.get("title", "")),
            "url": str(item.get("url", "")),
            "excerpt": sent.text.strip(),
            "lemma_set": lemmas,
        })

# 3) constitution des exemples: 5 phrases pour le target + 5 phrases par voisin
examples_map = defaultdict(list)
seen_excerpt = defaultdict(set)

for label, tgt in target_words.items():
    words_for_label = [tgt] + neighbors_by_target.get(label, [])
    for rec in sentence_records:
        for w in words_for_label:
            key = (label, w)
            if len(examples_map[key]) >= EXAMPLES_PER_WORD:
                continue
            if w not in rec["lemma_set"]:
                continue

            dedup_key = rec["excerpt"].lower()
            if dedup_key in seen_excerpt[key]:
                continue

            seen_excerpt[key].add(dedup_key)
            examples_map[key].append(rec)

rows = []
for label, tgt in target_words.items():
    words = [tgt] + neighbors_by_target.get(label, [])
    for w in words:
        role = "target" if w == tgt else "neighbor"
        ex_list = examples_map.get((label, w), [])
        if not ex_list:
            rows.append({
                "target_label": label,
                "target_word": tgt,
                "selected_word": w,
                "role": role,
                "example_id": 0,
                "excerpt": "",
                "title": "",
                "url": "",
                "n_examples_found": 0,
            })
            continue

        for i, ex in enumerate(ex_list, start=1):
            rows.append({
                "target_label": label,
                "target_word": tgt,
                "selected_word": w,
                "role": role,
                "example_id": i,
                "excerpt": ex["excerpt"],
                "title": ex["title"],
                "url": ex["url"],
                "n_examples_found": len(ex_list),
            })

w2v_context_examples_df = pd.DataFrame(rows)
print(f"Lignes construites: {len(w2v_context_examples_df)}")

Scan phrases (target + voisins):   0%|          | 0/1870 [00:00<?, ?it/s]

Lignes construites: 190


### 4.13. Sauvegarde des exemples Word2Vec

Les exemples contextuels et la liste des voisins sont sauvegardés afin de pouvoir les réexaminer ensuite dans une interface interactive.


In [54]:
w2v_context_examples_df.to_csv("discours_analysis_support/w2v_context_examples.csv", index=False)
with open("discours_analysis_support/neighbours_w2v.json", "w", encoding="utf-8") as f:
    json.dump(neighbors_by_target, f, ensure_ascii=False, indent=2)

### 4.14. Rechargement des exemples Word2Vec

Cette étape recharge les extraits contextualisés et les voisins sauvegardés. Elle permet de consulter les exemples sans refaire l’extraction.


In [55]:
w2v_context_examples_df = pd.read_csv("discours_analysis_support/w2v_context_examples.csv")
neighbors_by_target = json.load(open("discours_analysis_support/neighbours_w2v.json", "r", encoding="utf-8"))

### 4.15. Interface de consultation des exemples Word2Vec

L’interface compare les exemples du terme-cible et ceux du voisin sélectionné. Elle permet de vérifier concrètement dans quels contextes les deux mots apparaissent et d’éviter une interprétation purement automatique des similarités.


In [56]:
# 4) Visualisation interactive : target fixe + voisin sélectionné
if "w2v_context_examples_df" not in globals() or w2v_context_examples_df.empty:
    raise ValueError("w2v_context_examples_df est vide. Exécute la cellule précédente.")

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def _make_clickable(u: str) -> str:
    u = str(u).strip()
    return f'<a href="{u}" target="_blank">ouvrir</a>' if u else ""

def _prepare_examples_view(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    out = df.copy()
    out = out[out["example_id"] > 0].copy()
    out["url"] = out["url"].apply(_make_clickable)
    return out[["example_id", "excerpt", "title", "url"]]

target_dd = widgets.Dropdown(
    options=sorted(w2v_context_examples_df["target_label"].dropna().unique().tolist()),
    description="Target:",
    layout=widgets.Layout(width="320px"),
)

neighbor_dd = widgets.Dropdown(
    options=[],
    description="Voisin:",
    layout=widgets.Layout(width="360px"),
)

target_title_html = widgets.HTML()
target_table_html = widgets.HTML()
neighbor_title_html = widgets.HTML()
neighbor_table_html = widgets.HTML()

target_box = widgets.Box(
    [widgets.VBox([target_title_html, target_table_html])],
    layout=widgets.Layout(
        width="100%",
        border="1px solid #ddd",
        padding="8px",
        overflow="auto"
    )
)

neighbor_box = widgets.Box(
    [widgets.VBox([neighbor_title_html, neighbor_table_html])],
    layout=widgets.Layout(
        width="100%",
        border="1px solid #ddd",
        padding="8px",
        overflow="auto"
    )
)

updating_ui = False

def update_neighbors(*_):
    global updating_ui
    updating_ui = True
    try:
        label = target_dd.value
        opts = neighbors_by_target.get(label, [])
        new_options = opts if opts else ["(aucun voisin)"]
        prev = neighbor_dd.value
        neighbor_dd.options = new_options
        neighbor_dd.value = prev if prev in new_options else new_options[0]
    finally:
        updating_ui = False

def render_context(*_):
    if updating_ui:
        return

    label = target_dd.value
    tgt = target_words[label]
    neigh = neighbor_dd.value

    target_title_html.value = f"<b>Target: {label} ({tgt})</b><br><br>5 phrases avec le target:"
    target_df = _prepare_examples_view(
        w2v_context_examples_df[
            (w2v_context_examples_df["target_label"] == label)
            & (w2v_context_examples_df["selected_word"] == tgt)
        ]
    )

    if target_df.empty:
        target_table_html.value = "<i>Aucune phrase trouvée pour le target.</i>"
    else:
        target_table_html.value = target_df.head(EXAMPLES_PER_WORD).to_html(index=False, escape=False)

    neighbor_title_html.value = "<b>5 phrases avec le voisin sélectionné:</b>"

    if neigh == "(aucun voisin)":
        neighbor_table_html.value = "<i>Aucun voisin disponible pour ce target.</i>"
        return

    neigh_df = _prepare_examples_view(
        w2v_context_examples_df[
            (w2v_context_examples_df["target_label"] == label)
            & (w2v_context_examples_df["selected_word"] == neigh)
        ]
    )

    if neigh_df.empty:
        neighbor_table_html.value = "<i>Aucune phrase trouvée pour ce voisin.</i>"
    else:
        neighbor_table_html.value = neigh_df.head(EXAMPLES_PER_WORD).to_html(index=False, escape=False)

def on_target_change(change):
    if updating_ui:
        return
    update_neighbors()
    render_context()

def on_neighbor_change(change):
    if updating_ui:
        return
    render_context()

target_dd.observe(on_target_change, names="value")
neighbor_dd.observe(on_neighbor_change, names="value")

ui = widgets.VBox([
    widgets.HBox([target_dd, neighbor_dd], layout=widgets.Layout(gap="12px")),
    target_box,
    neighbor_box,
])

update_neighbors()
render_context()
display(ui)

## 5. Analyse des patrons syntaxiques avec spaCy

Cette dernière grande section examine les relations syntaxiques dans lesquelles apparaissent les termes-cibles. L’objectif est de dépasser la simple proximité lexicale pour observer les rôles grammaticaux associés aux cibles : sujet d’un verbe, objet d’un verbe ou nom qualifié par un adjectif.


In [104]:
# Step 3 — Dependency patterns: target comme sujet / objet + adjectifs fréquents
if "nlp" not in globals():
    raise ValueError("nlp absent. Exécute d'abord la partie spaCy.")
if "data" not in globals() or not data:
    raise ValueError("data absent ou vide.")
if "TARGETS_W2V" not in globals():
    raise ValueError("TARGETS_W2V absent. Exécute d'abord la partie Word2Vec.")

### 5.1. Initialisation des cibles et des compteurs syntaxiques

Cette cellule prépare les structures de stockage nécessaires pour compter les verbes et adjectifs associés aux termes-cibles. Chaque cible est reliée à son lemme russe normalisé.


In [105]:
def _norm_dep(x: str) -> str:
    if "normalize_token" in globals():
        return normalize_token(x)
    return str(x).lower().replace("ё", "е").strip()

target_lemma_to_label = {_norm_dep(v): k for k, v in TARGETS_W2V.items()}
subject_verbs = defaultdict(Counter)   # target = sujet
object_verbs = defaultdict(Counter)    # target = objet
target_adjs = defaultdict(Counter)     # adjectifs liés au target

### 5.2. Extraction des relations syntaxiques et des exemples

Le code parcourt les articles avec spaCy et identifie les cas où les termes-cibles apparaissent comme sujets, objets ou avec des adjectifs liés. Pour chaque relation fréquente, plusieurs exemples textuels sont conservés.

Cette étape permet d’observer non seulement les mots associés aux cibles, mais aussi les actions, qualités et positions grammaticales qui leur sont attribuées dans les articles.


In [109]:
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm

# ----------------------------
# Helpers
# ----------------------------
def _safe_sent_text(tok):
    try:
        return tok.sent.text.strip()
    except Exception:
        return tok.doc.text.strip()

def _clean_spaces(txt: str) -> str:
    return " ".join(str(txt).split())

def _format_example(sent_text: str, title: str = "", url: str = "") -> str:
    sent_text = _clean_spaces(sent_text)
    parts = [sent_text]
    meta = []
    if title:
        meta.append(f"Titre: {title}")
    if url:
        meta.append(f"URL: {url}")
    if meta:
        parts.append(" | ".join(meta))
    return " || ".join(parts)

def _add_example(example_store, seen_store, key, example_text, max_examples=3):
    if example_text in seen_store[key]:
        return
    if len(example_store[key]) >= max_examples:
        return
    seen_store[key].add(example_text)
    example_store[key].append(example_text)

# ----------------------------
# Recompute from scratch
# ----------------------------
subject_verbs = defaultdict(Counter)   # target = sujet
object_verbs = defaultdict(Counter)    # target = objet
target_adjs = defaultdict(Counter)     # adjectifs liés au target

# examples stores
subject_examples = defaultdict(list)
object_examples = defaultdict(list)
adj_examples = defaultdict(list)

# anti-duplication
subject_seen = defaultdict(set)
object_seen = defaultdict(set)
adj_seen = defaultdict(set)

for item in tqdm(data, desc="Dependency patterns (targets + examples)"):
    txt = str(item.get("text", "")).strip()
    if not txt:
        continue

    title = str(item.get("title", "")).strip()
    url = str(item.get("url", "")).strip()

    doc = nlp(txt)

    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        lemma = _norm_dep(tok.lemma_)
        if lemma not in target_lemma_to_label:
            continue

        label = target_lemma_to_label[lemma]
        sent_text = _safe_sent_text(tok)

        # ----------------------------
        # adjectifs liés directement au target
        # ----------------------------
        for ch in tok.children:
            if ch.pos_ == "ADJ" and ch.dep_ in {"amod", "acl", "appos"}:
                adj_lemma = _norm_dep(ch.lemma_)
                target_adjs[label][adj_lemma] += 1

                key = (label, "adjectif lié au target", adj_lemma)
                ex = _format_example(sent_text, title, url)
                _add_example(adj_examples, adj_seen, key, ex, max_examples=3)

        # ----------------------------
        # target comme sujet du verbe
        # ----------------------------
        if tok.dep_.startswith("nsubj") and tok.head.pos_ in {"VERB", "AUX"}:
            verb_lemma = _norm_dep(tok.head.lemma_)
            subject_verbs[label][verb_lemma] += 1

            key = (label, "target = sujet → verbe", verb_lemma)
            ex = _format_example(sent_text, title, url)
            _add_example(subject_examples, subject_seen, key, ex, max_examples=3)

        # ----------------------------
        # target comme objet du verbe
        # ----------------------------
        if (tok.dep_.startswith("obj") or tok.dep_ == "iobj") and tok.head.pos_ in {"VERB", "AUX"}:
            verb_lemma = _norm_dep(tok.head.lemma_)
            object_verbs[label][verb_lemma] += 1

            key = (label, "target = objet → verbe", verb_lemma)
            ex = _format_example(sent_text, title, url)
            _add_example(object_examples, object_seen, key, ex, max_examples=3)

# ----------------------------
# Build rows with examples
# ----------------------------
TOP_N = 25
rows_subj, rows_obj, rows_adj = [], [], []

for label in TARGETS_W2V.keys():

    for verb, freq in subject_verbs[label].most_common(TOP_N):
        key = (label, "target = sujet → verbe", verb)
        exs = subject_examples.get(key, [])
        rows_subj.append({
            "target": label,
            "type": "target = sujet → verbe",
            "token": verb,
            "freq": int(freq),
            "example_1": exs[0] if len(exs) > 0 else "",
            "example_2": exs[1] if len(exs) > 1 else "",
            "example_3": exs[2] if len(exs) > 2 else "",
        })

    for verb, freq in object_verbs[label].most_common(TOP_N):
        key = (label, "target = objet → verbe", verb)
        exs = object_examples.get(key, [])
        rows_obj.append({
            "target": label,
            "type": "target = objet → verbe",
            "token": verb,
            "freq": int(freq),
            "example_1": exs[0] if len(exs) > 0 else "",
            "example_2": exs[1] if len(exs) > 1 else "",
            "example_3": exs[2] if len(exs) > 2 else "",
        })

    for adj, freq in target_adjs[label].most_common(TOP_N):
        key = (label, "adjectif lié au target", adj)
        exs = adj_examples.get(key, [])
        rows_adj.append({
            "target": label,
            "type": "adjectif lié au target",
            "token": adj,
            "freq": int(freq),
            "example_1": exs[0] if len(exs) > 0 else "",
            "example_2": exs[1] if len(exs) > 1 else "",
            "example_3": exs[2] if len(exs) > 2 else "",
        })

# ----------------------------
# Final tables
# ----------------------------
dep_patterns_df = pd.DataFrame(rows_subj + rows_obj + rows_adj)

target_subject_verbs_df = pd.DataFrame(rows_subj)
target_object_verbs_df = pd.DataFrame(rows_obj)
target_adjectives_df = pd.DataFrame(rows_adj)

display(dep_patterns_df.head(10))

Dependency patterns (targets + examples):   0%|          | 0/1870 [00:00<?, ?it/s]

,target,type,token,freq,example_1,example_2,example_3
0,Refugies,target = sujet → verbe,прибыть,44,"Чеченское отделение партии, при поддержке Реги...",Как сообщил журналистам руководитель пресс-слу...,"Ранее в понедельник в МВД ФРГ сообщили, что в ..."
1,Refugies,target = sujet → verbe,прибывать,10,"По его словам, в ближайшие дни в регион начнут...","""В Подмосковье продолжают прибывать беженцы из...",В настоящий момент беженцы из ДНР и ЛНР на Кам...
2,Refugies,target = sujet → verbe,мочь,9,Замглавы МЧС России Виктор Яцуценко в субботу ...,Городок жизнеобеспечения состоит из больших на...,Ранее замглавы МЧС России Виктор Яцуценко сооб...
3,Refugies,target = sujet → verbe,находиться,9,"""На территории страны уже находятся беженцы из...","""В 125 пунктах временного размещения в Ростовс...","Также он сообщил, что в пунктах временного раз..."
4,Refugies,target = sujet → verbe,отправляться,6,"Некоторые беженцы из третьих стран, в частност...","Некоторые беженцы из третьих стран, в частност...","Некоторые беженцы из третьих стран, в частност..."
5,Refugies,target = sujet → verbe,обеспечить,5,"В пресс-службе отметили, что беженцы обеспечен...","Как сообщается, беженцы будут обеспечены горяч...","Плаксин отметил, что беженцы из Донбасса, прож..."
6,Refugies,target = sujet → verbe,сдавать,4,"/ТАСС/. Беженцы, прибывающие в Ростовскую обла...",Накануне глава Роспотребнадзора Анна Попова со...,"Накануне Попова сообщила, что беженцы, прибыва..."
7,Refugies,target = sujet → verbe,смочь,4,"/ТАСС/. Американское руководство рассчитывает,...",/ТАСС/. Беженцы с Украины смогут оставаться на...,"Прием будет вестись очно, а обратиться смогут ..."
8,Refugies,target = sujet → verbe,проживать,4,"В пансионате ""Царицыно озеро"" сейчас проживают...",/ТАСС/. Порядка 200 беженцев из Артемовска (ук...,"Белецкая уточнила, что в населенном пункте про..."
9,Refugies,target = sujet → verbe,включить,4,В остальных государствах это число не превышае...,В остальных государствах это число не превышае...,В национальные программы защиты и поддержки вк...


### 5.3. Sauvegarde des patrons syntaxiques

Les trois tableaux produits — verbes où la cible est sujet, verbes où la cible est objet et adjectifs associés — sont sauvegardés séparément. Ces fichiers constituent un support pour l’analyse qualitative des rôles discursifs.


In [122]:
target_subject_verbs_df.to_csv("discours_analysis_support/dep_patterns_subject_verbs.csv", index=False)
target_object_verbs_df.to_csv("discours_analysis_support/dep_patterns_object_verbs.csv", index=False)
target_adjectives_df.to_csv("discours_analysis_support/dep_patterns_adjectives.csv", index=False)

### 5.4. Rechargement des tableaux syntaxiques

Cette cellule recharge les résultats syntaxiques sauvegardés afin de pouvoir les explorer sans relancer le traitement spaCy sur tout le corpus.


In [ ]:
target_subject_verbs_df = pd.read_csv("discours_analysis_support/dep_patterns_subject_verbs.csv")
target_object_verbs_df = pd.read_csv("discours_analysis_support/dep_patterns_object_verbs.csv")
target_adjectives_df = pd.read_csv("discours_analysis_support/dep_patterns_adjectives.csv")

### 5.5. Table interactive des patrons syntaxiques

L’interface permet de naviguer entre les trois types de résultats syntaxiques, de filtrer par cible, par token ou par contenu des exemples, puis de consulter les extraits associés.


In [ ]:

import math
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

required = ["target_subject_verbs_df", "target_object_verbs_df", "target_adjectives_df"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Variables manquantes: {missing}. Exécute d'abord la cellule Step 3.")

# --- close previous UI if this cell is re-run
_prev_state = globals().get("_step3_ui_state", {})
for _w in _prev_state.get("widgets", []):
    try:
        _w.close()
    except Exception:
        pass

def _normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    if "verb" in d.columns:
        d = d.rename(columns={"verb": "token"})
    if "adjective" in d.columns:
        d = d.rename(columns={"adjective": "token"})

    # colonnes minimales
    if "target" not in d.columns:
        d["target"] = ""
    if "type" not in d.columns:
        d["type"] = ""
    if "token" not in d.columns:
        d["token"] = ""
    if "freq" not in d.columns:
        d["freq"] = 0

    # exemples optionnels
    for col in ["example_1", "example_2", "example_3"]:
        if col not in d.columns:
            d[col] = ""

    cols = ["target", "type", "token", "freq", "example_1", "example_2", "example_3"]
    d = d[cols].copy()

    d["target"] = d["target"].astype(str)
    d["type"] = d["type"].astype(str)
    d["token"] = d["token"].astype(str)
    d["freq"] = pd.to_numeric(d["freq"], errors="coerce").fillna(0).astype(int)

    for col in ["example_1", "example_2", "example_3"]:
        d[col] = d[col].fillna("").astype(str)

    return d

table_map = {
    "target_subject_verbs_df": _normalize_df(target_subject_verbs_df),
    "target_object_verbs_df": _normalize_df(target_object_verbs_df),
    "target_adjectives_df": _normalize_df(target_adjectives_df),
}

dataset_dd = widgets.Dropdown(
    options=list(table_map.keys()),
    value="target_subject_verbs_df",
    description="Dataset:",
    layout=widgets.Layout(width="340px"),
)

target_dd = widgets.Dropdown(
    options=["Tous"],
    value="Tous",
    description="Target:",
    layout=widgets.Layout(width="260px"),
)

token_text = widgets.Text(
    value="",
    placeholder="rechercher token...",
    description="Token:",
    layout=widgets.Layout(width="300px"),
)

example_text = widgets.Text(
    value="",
    placeholder="rechercher dans les exemples...",
    description="Exemple:",
    layout=widgets.Layout(width="340px"),
)

sort_dd = widgets.Dropdown(
    options=["freq", "target", "type", "token"],
    value="freq",
    description="Tri:",
    layout=widgets.Layout(width="180px"),
)

asc_cb = widgets.Checkbox(
    value=False,
    description="Croissant",
    indent=False,
    layout=widgets.Layout(width="120px"),
)

page_size_dd = widgets.Dropdown(
    options=[10, 20, 30, 50, 100],
    value=20,
    description="Lignes:",
    layout=widgets.Layout(width="170px"),
)

page_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="Page:",
    continuous_update=False,
    layout=widgets.Layout(width="340px"),
)

info_html = widgets.HTML()
table_html = widgets.HTML()

_is_refreshing = False

def _current_df() -> pd.DataFrame:
    return table_map[dataset_dd.value].copy()

def _format_examples_cell(row):
    parts = []
    for i in [1, 2, 3]:
        ex = row.get(f"example_{i}", "")
        if ex:
            parts.append(f"<div style='margin-bottom:6px;'><b>{i}.</b> {ex}</div>")
    return "".join(parts)

def refresh(*args):
    global _is_refreshing
    if _is_refreshing:
        return

    _is_refreshing = True
    try:
        d = _current_df()

        available_targets = ["Tous"] + sorted(d["target"].dropna().unique().tolist())
        current_target = target_dd.value

        if list(target_dd.options) != available_targets:
            target_dd.options = available_targets

        if current_target not in available_targets:
            target_dd.value = "Tous"
            current_target = "Tous"

        if current_target != "Tous":
            d = d[d["target"] == current_target]

        q = token_text.value.strip().lower()
        if q:
            d = d[d["token"].str.lower().str.contains(q, na=False)]

        q_ex = example_text.value.strip().lower()
        if q_ex:
            mask = (
                d["example_1"].str.lower().str.contains(q_ex, na=False) |
                d["example_2"].str.lower().str.contains(q_ex, na=False) |
                d["example_3"].str.lower().str.contains(q_ex, na=False)
            )
            d = d[mask]

        d = d.sort_values(sort_dd.value, ascending=asc_cb.value).reset_index(drop=True)

        total = len(d)
        page_size = int(page_size_dd.value)
        total_pages = max(1, math.ceil(total / page_size))

        if page_slider.max != total_pages:
            page_slider.max = total_pages
        if page_slider.value > total_pages:
            page_slider.value = total_pages

        p = page_slider.value
        start = (p - 1) * page_size
        end = start + page_size
        page_df = d.iloc[start:end].copy()

        info_html.value = (
            f"<b>Dataset:</b> {dataset_dd.value} "
            f"&nbsp;&nbsp; <b>Target:</b> {current_target}"
        )

        if page_df.empty:
            table_html.value = "<p><i>Aucune ligne pour ce filtre.</i></p>"
        else:
            # готовим колонку с красиво собранными примерами
            page_df["examples"] = page_df.apply(_format_examples_cell, axis=1)

            display_cols = ["target", "type", "token", "freq", "examples"]
            html_df = page_df[display_cols].copy()

            table_css = """
            <style>
            .dep-table {
                border-collapse: collapse;
                width: 100%;
                table-layout: fixed;
                font-size: 13px;
            }
            .dep-table th, .dep-table td {
                border: 1px solid #ddd;
                padding: 8px;
                vertical-align: top;
                text-align: left;
                word-wrap: break-word;
                white-space: normal;
            }
            .dep-table th {
                background: #f5f5f5;
            }
            .dep-table th:nth-child(1), .dep-table td:nth-child(1) { width: 11%; }
            .dep-table th:nth-child(2), .dep-table td:nth-child(2) { width: 18%; }
            .dep-table th:nth-child(3), .dep-table td:nth-child(3) { width: 12%; }
            .dep-table th:nth-child(4), .dep-table td:nth-child(4) { width: 6%; }
            .dep-table th:nth-child(5), .dep-table td:nth-child(5) { width: 53%; }
            </style>
            """

            footer = (
                f"<p style='margin-top:8px;'>"
                f"Lignes affichées: {start + 1}-{min(end, total)} / {total} "
                f"| Page {p}/{total_pages}</p>"
            )

            table_html.value = (
                table_css +
                html_df.to_html(index=False, escape=False, classes="dep-table") +
                footer
            )

    finally:
        _is_refreshing = False

for w in [dataset_dd, target_dd, token_text, example_text, sort_dd, asc_cb, page_size_dd, page_slider]:
    w.observe(refresh, names="value")

ui = widgets.VBox([
    widgets.HTML("<b>Step 3 — Interactive table (v5, with examples)</b>"),
    widgets.HBox([dataset_dd, target_dd, token_text], layout=widgets.Layout(gap="12px")),
    widgets.HBox([example_text, sort_dd, asc_cb], layout=widgets.Layout(gap="12px")),
    widgets.HBox([page_size_dd, page_slider], layout=widgets.Layout(gap="12px")),
    info_html,
    table_html,
])

globals()["_step3_ui_state"] = {
    "widgets": [
        dataset_dd, target_dd, token_text, example_text, sort_dd, asc_cb,
        page_size_dd, page_slider, info_html, table_html, ui
    ]
}

clear_output(wait=True)
display(ui)
refresh()

### 5.6. Synthèse quantitative des occurrences syntaxiques

Cette dernière cellule calcule des indicateurs généraux pour chaque terme-cible : nombre de documents et de phrases où il apparaît, nombre total de mentions, fréquence des usages comme sujet ou objet, et présence d’adjectifs associés.

Ces indicateurs donnent une vue synthétique de la place occupée par chaque terme dans le corpus et complètent les analyses qualitatives précédentes.


In [111]:
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

# Проверяем, что нужные переменные есть
required = ["data", "nlp", "target_lemma_to_label", "_norm_dep"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Variables manquantes: {missing}")

# Хранилища
docs_with_target = defaultdict(set)      # label -> {doc_id}
sents_with_target = defaultdict(set)     # label -> {(doc_id, sent_id)}
target_mentions = defaultdict(int)       # label -> total target mentions

target_as_subject = defaultdict(int)     # label -> target tokens used as subject
target_as_object = defaultdict(int)      # label -> target tokens used as object
target_with_adj_token = defaultdict(int) # label -> target tokens with >=1 linked adjective
adj_relations = defaultdict(int)         # label -> total adjective relations

total_docs = 0
total_sents = 0

for doc_id, item in enumerate(tqdm(data, desc="Target statistics")):
    txt = str(item.get("text", "")).strip()
    if not txt:
        continue

    total_docs += 1
    doc = nlp(txt)

    # создаем карту token.i -> sent_id
    sent_id_by_token_i = {}
    for sent_id, sent in enumerate(doc.sents):
        total_sents += 1
        for tok in sent:
            sent_id_by_token_i[tok.i] = sent_id

    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        lemma = _norm_dep(tok.lemma_)
        if lemma not in target_lemma_to_label:
            continue

        label = target_lemma_to_label[lemma]
        sent_id = sent_id_by_token_i.get(tok.i, -1)

        # 1) docs / sents containing target
        docs_with_target[label].add(doc_id)
        sents_with_target[label].add((doc_id, sent_id))

        # 2) total target mentions
        target_mentions[label] += 1

        # 3) syntactic relations
        if tok.dep_.startswith("nsubj") and tok.head.pos_ in {"VERB", "AUX"}:
            target_as_subject[label] += 1

        if (tok.dep_.startswith("obj") or tok.dep_ == "iobj") and tok.head.pos_ in {"VERB", "AUX"}:
            target_as_object[label] += 1

        linked_adj_count = 0
        for ch in tok.children:
            if ch.pos_ == "ADJ" and ch.dep_ in {"amod", "acl", "appos"}:
                linked_adj_count += 1

        if linked_adj_count > 0:
            target_with_adj_token[label] += 1
            adj_relations[label] += linked_adj_count

# Собираем итоговую таблицу
labels = sorted(set(target_lemma_to_label.values()))

rows = []
for label in labels:
    n_docs = len(docs_with_target[label])
    n_sents = len(sents_with_target[label])
    mentions = target_mentions[label]

    rows.append({
        "target": label,
        "docs_with_target": n_docs,
        "docs_with_target_pct": round(100 * n_docs / total_docs, 2) if total_docs else 0,
        "sents_with_target": n_sents,
        "sents_with_target_pct": round(100 * n_sents / total_sents, 2) if total_sents else 0,
        "target_mentions": mentions,
        "mentions_per_doc_with_target": round(mentions / n_docs, 2) if n_docs else 0,
        "target_as_subject": target_as_subject[label],
        "target_as_object": target_as_object[label],
        "target_with_adj_token": target_with_adj_token[label],
        "adj_relations": adj_relations[label],
    })

target_stats_df = pd.DataFrame(rows).sort_values("target").reset_index(drop=True)

display(target_stats_df)

Target statistics:   0%|          | 0/1870 [00:00<?, ?it/s]

,target,docs_with_target,docs_with_target_pct,sents_with_target,sents_with_target_pct,target_mentions,mentions_per_doc_with_target,target_as_subject,target_as_object,target_with_adj_token,adj_relations
0,Benevoles,187,10.00,319,1.50,327,1.75,152,19,17,18
1,Refugies,659,35.24,1521,7.15,1544,2.34,160,396,96,96
2,Russie,1203,64.33,2755,12.96,2841,2.36,79,7,222,231
3,Ukraine,1024,54.76,2082,9.79,2152,2.10,13,42,6,8
